# Evaluación Parcial N°3 - Proyecto end-to-end de análisis de datos

**Asignatura:** SCY1101 - Programación para la Ciencia de Datos  
**Proyecto:** Pipeline ETL, API, dashboard, Git y Docker para análisis de actividad física por hora.

Este notebook está preparado para Google Colab y cubre los requisitos del encargo: integración de al menos tres fuentes de datos, pipeline ETL automatizado, dashboard interactivo, documentación técnica, uso profesional de Git y despliegue con Docker.

## Matriz de cumplimiento de la rúbrica

| Indicador | Evidencia incluida |
|---|---|
| Pipeline ETL robusto | CSV + API REST + SQLite, validación de esquemas, logging, manejo de errores, lectura por chunks y dataset final. |
| Documentación completa | README, arquitectura, API, manual de usuario, guía de despliegue y guion de presentación. |
| Dashboard interactivo | App Streamlit con Plotly y vistas ejecutiva, técnica y operativa. |
| Git profesional | Flujo de ramas, commits, issues, pull requests y evidencia colaborativa. |
| Docker | Dockerfiles, docker-compose y variables de entorno. |

In [1]:
# 1. Instalación de librerías
!pip -q install pandas numpy requests plotly streamlit fastapi uvicorn sqlalchemy python-dotenv pytest scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 90.4 MB/s eta 0:00:00


In [2]:
# 2. Imports y estructura de proyecto
import os, json, shutil, sqlite3, logging, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/content/proyecto_etl_actividad')
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
ETL_DIR = PROJECT_ROOT / 'etl'
DASH_DIR = PROJECT_ROOT / 'dashboards'
API_DIR = PROJECT_ROOT / 'api'
DOCS_DIR = PROJECT_ROOT / 'docs'
DOCKER_DIR = PROJECT_ROOT / 'docker'
TESTS_DIR = PROJECT_ROOT / 'tests'
REPO_DIR = PROJECT_ROOT / 'repo'
LOGS_DIR = PROJECT_ROOT / 'logs'

for carpeta in [DATA_RAW, DATA_PROCESSED, ETL_DIR, DASH_DIR, API_DIR, DOCS_DIR, DOCKER_DIR, TESTS_DIR, REPO_DIR, LOGS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOGS_DIR / 'etl_pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print('Proyecto creado en:', PROJECT_ROOT)

Proyecto creado en: /content/proyecto_etl_actividad


## 1. Carga de archivos CSV
Sube a Colab estos archivos:

- `hourlySteps_sucio.csv`
- `hourlySteps_merged.csv`
- `hourlySteps_clean(1).csv`

El archivo principal del ETL es `hourlySteps_sucio.csv`, porque permite demostrar limpieza, transformación y validaciones.

In [3]:
# 3. Carga de archivos desde el entorno de ejecución de Colab usando pd.read_csv
# Antes de ejecutar esta celda, sube los CSV al panel izquierdo de Colab en:
# /content/
# o crea la carpeta /content/data/raw/ y déjalos ahí.
# No se usa files.upload().

archivos_esperados = [
    'hourlySteps_sucio.csv',
    'hourlySteps_merged.csv',
    'hourlySteps_clean(1).csv'
]

rutas_busqueda = [
    Path('/content'),
    Path('/content/data/raw'),
    DATA_RAW,
    Path.cwd()
]

def buscar_archivo(nombre_archivo):
    for carpeta in rutas_busqueda:
        ruta = carpeta / nombre_archivo
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f'No se encontró {nombre_archivo}. Súbelo al entorno de Colab en /content/ o /content/data/raw/'
    )

rutas_archivos = {}
for nombre in archivos_esperados:
    ruta_encontrada = buscar_archivo(nombre)

    # Lectura con pd.read_csv para verificar que el archivo abre correctamente.
    df_temporal = pd.read_csv(ruta_encontrada)
    print(f'{nombre}: cargado correctamente desde {ruta_encontrada} | filas={df_temporal.shape[0]} | columnas={df_temporal.shape[1]}')

    # Copia interna para mantener la estructura profesional del proyecto.
    destino = DATA_RAW / nombre
    if ruta_encontrada.resolve() != destino.resolve():
        shutil.copy(ruta_encontrada, destino)
    rutas_archivos[nombre] = destino

print('\nArchivos listos en data/raw para continuar el pipeline ETL.')


hourlySteps_sucio.csv: cargado correctamente desde /content/hourlySteps_sucio.csv | filas=28901 | columnas=3
hourlySteps_merged.csv: cargado correctamente desde /content/hourlySteps_merged.csv | filas=24084 | columnas=3
hourlySteps_clean(1).csv: cargado correctamente desde /content/hourlySteps_clean(1).csv | filas=21228 | columnas=7

Archivos listos en data/raw para continuar el pipeline ETL.


In [4]:
# 4. Verificación de archivos
archivos_requeridos = ['hourlySteps_sucio.csv', 'hourlySteps_merged.csv', 'hourlySteps_clean(1).csv']
for archivo in archivos_requeridos:
    ruta = DATA_RAW / archivo
    print(archivo, 'OK' if ruta.exists() else 'FALTA')

hourlySteps_sucio.csv OK
hourlySteps_merged.csv OK
hourlySteps_clean(1).csv OK


## 2. Fuente 1: CSV sucio
Primera fuente de datos: archivo CSV con pasos por hora.

In [5]:
# 5. Extracción desde CSV sucio
ruta_csv_sucio = rutas_archivos.get('hourlySteps_sucio.csv', DATA_RAW / 'hourlySteps_sucio.csv')
df_sucio = pd.read_csv(ruta_csv_sucio)

print('Dimensiones:', df_sucio.shape)
display(df_sucio.head())
print(df_sucio.info())


Dimensiones: (28901, 3)


,Id,ActivityHour,StepTotal
0,1.503960e+09,3/12/2016 12:00:00 AM,0.0
1,1.503960e+09,3/12/2016 1:00:00 AM,NaN
2,1.503960e+09,3/12/2016 2:00:00 AM,0.0
3,1.503960e+09,3/12/2016 3:00:00 AM,0.0
4,1.503960e+09,3/12/2016 4:00:00 AM,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28901 entries, 0 to 28900
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            27456 non-null  float64
 1   ActivityHour  27456 non-null  object 
 2   StepTotal     27456 non-null  float64
dtypes: float64(2), object(1)
memory usage: 677.5+ KB
None


In [6]:
# 6. Diagnóstico de calidad
def diagnostico_calidad(df, nombre='dataset'):
    resumen = pd.DataFrame({
        'columna': df.columns,
        'tipo_dato': df.dtypes.astype(str).values,
        'nulos': df.isna().sum().values,
        'porcentaje_nulos': (df.isna().mean() * 100).round(2).values,
        'unicos': df.nunique(dropna=True).values
    })
    print(f'Dataset: {nombre}')
    print('Filas duplicadas:', df.duplicated().sum())
    return resumen

display(diagnostico_calidad(df_sucio, 'hourlySteps_sucio'))

Dataset: hourlySteps_sucio
Filas duplicadas: 4444


,columna,tipo_dato,nulos,porcentaje_nulos,unicos
0,Id,float64,1445,5.0,34
1,ActivityHour,object,1445,5.0,755
2,StepTotal,float64,1445,5.0,2179


## 3. Validación de esquemas
Se valida que las columnas obligatorias existan y que sus tipos sean aceptables antes de transformar.

In [7]:
# 7. Validación de esquema
ESQUEMA_STEPS = {
    'Id': ['int64', 'float64', 'object'],
    'ActivityHour': ['object', 'datetime64[ns]'],
    'StepTotal': ['int64', 'float64', 'object']
}

def validar_esquema(df, esquema, nombre_fuente='fuente'):
    errores = []
    for columna, tipos_permitidos in esquema.items():
        if columna not in df.columns:
            errores.append(f'Falta la columna obligatoria: {columna}')
        else:
            tipo_actual = str(df[columna].dtype)
            if tipo_actual not in tipos_permitidos:
                errores.append(f'Columna {columna} tiene tipo {tipo_actual}; esperado: {tipos_permitidos}')
    if errores:
        logging.error(f'Errores de esquema en {nombre_fuente}: {errores}')
        raise ValueError(' | '.join(errores))
    logging.info(f'Esquema validado correctamente para {nombre_fuente}')
    return True

validar_esquema(df_sucio, ESQUEMA_STEPS, 'hourlySteps_sucio')
print('Esquema validado correctamente')

Esquema validado correctamente


## 4. Fuente 2: API REST
Segunda fuente de datos: API REST de clima con Open-Meteo. Si la API falla, el código crea datos de respaldo para que la demo no se detenga.

In [8]:
# 8. Extracción desde API REST con manejo de errores
def extraer_clima_api(start_date='2016-03-12', end_date='2016-04-12', lat=-33.45, lon=-70.66):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,precipitation',
        'timezone': 'America/Santiago'
    }
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        hourly = response.json().get('hourly', {})
        df_api = pd.DataFrame({
            'ActivityHour': hourly.get('time', []),
            'temperature_2m': hourly.get('temperature_2m', []),
            'precipitation': hourly.get('precipitation', [])
        })
        df_api['ActivityHour'] = pd.to_datetime(df_api['ActivityHour'], errors='coerce')
        df_api['fuente_clima'] = 'api_open_meteo'
        logging.info('Clima extraído correctamente desde API REST')
        return df_api
    except Exception as e:
        logging.warning(f'Fallo API REST. Se usará fallback local. Error: {e}')
        fechas = pd.date_range(start=start_date, end=end_date, freq='H')
        np.random.seed(42)
        return pd.DataFrame({
            'ActivityHour': fechas,
            'temperature_2m': np.random.normal(18, 5, len(fechas)).round(1),
            'precipitation': np.random.choice([0, 0, 0, 0.2, 1.5], len(fechas)),
            'fuente_clima': 'fallback_generado'
        })

df_clima = extraer_clima_api()
print('Dimensiones API clima:', df_clima.shape)
display(df_clima.head())

Dimensiones API clima: (768, 4)


,ActivityHour,temperature_2m,precipitation,fuente_clima
0,2016-03-12 00:00:00,19.1,0.0,api_open_meteo
1,2016-03-12 01:00:00,19.0,0.0,api_open_meteo
2,2016-03-12 02:00:00,18.8,0.0,api_open_meteo
3,2016-03-12 03:00:00,18.1,0.0,api_open_meteo
4,2016-03-12 04:00:00,17.7,0.0,api_open_meteo


## 5. Fuente 3: Base de datos SQL
Tercera fuente de datos: SQLite local con segmentos de usuario, metas y clasificación operativa.

In [9]:
# 9. Creación y extracción desde SQLite
ruta_db = DATA_RAW / 'actividad_usuarios.db'
conn = sqlite3.connect(ruta_db)

ids_validos = df_sucio['Id'].dropna().astype('int64').drop_duplicates().head(100)
np.random.seed(42)

df_usuarios_sql = pd.DataFrame({
    'Id': ids_validos,
    'segmento_usuario': np.random.choice(['bajo', 'medio', 'alto'], size=len(ids_validos), p=[0.35, 0.45, 0.20]),
    'meta_pasos_hora': np.random.choice([250, 500, 750, 1000], size=len(ids_validos)),
    'audiencia_operativa': np.random.choice(['seguimiento', 'riesgo_inactividad', 'usuario_activo'], size=len(ids_validos))
})

df_usuarios_sql.to_sql('usuarios', conn, if_exists='replace', index=False)
df_sql = pd.read_sql_query('SELECT * FROM usuarios', conn)
conn.close()

print('Dimensiones SQL:', df_sql.shape)
display(df_sql.head())

Dimensiones SQL: (34, 4)


,Id,segmento_usuario,meta_pasos_hora,audiencia_operativa
0,1503960366,medio,1000,riesgo_inactividad
1,1624580081,alto,500,riesgo_inactividad
2,1644430081,medio,500,riesgo_inactividad
3,1844505072,medio,500,riesgo_inactividad
4,1927972279,bajo,250,riesgo_inactividad


## 6. Transformación robusta
Incluye conversión de tipos, eliminación de claves nulas, imputación controlada, eliminación de duplicados, corrección de inconsistencias y creación de variables analíticas.

In [10]:
# 10. Transformación robusta del dataset principal
def transformar_steps(df):
    try:
        df = df.copy()
        validar_esquema(df, ESQUEMA_STEPS, 'steps_transformacion')
        filas_iniciales = len(df)

        df['Id'] = pd.to_numeric(df['Id'], errors='coerce')
        df['StepTotal'] = pd.to_numeric(df['StepTotal'], errors='coerce')
        df['ActivityHour'] = pd.to_datetime(df['ActivityHour'], errors='coerce')

        df = df.dropna(subset=['Id', 'ActivityHour'])
        df['StepTotal'] = df['StepTotal'].fillna(0).clip(lower=0)
        df['Id'] = df['Id'].astype('int64')
        df['StepTotal'] = df['StepTotal'].astype(float)
        df = df.drop_duplicates(subset=['Id', 'ActivityHour'], keep='first')

        df['Hora'] = df['ActivityHour'].dt.hour
        df['Fecha'] = df['ActivityHour'].dt.date.astype(str)
        df['Dia'] = df['ActivityHour'].dt.day_name()
        df['Mes'] = df['ActivityHour'].dt.month
        df['FinDeSemana'] = df['ActivityHour'].dt.dayofweek.isin([5, 6]).astype(int)
        df['PeriodoDia'] = pd.cut(df['Hora'], bins=[-1, 5, 11, 17, 23], labels=['madrugada', 'mañana', 'tarde', 'noche']).astype(str)

        scaler = StandardScaler()
        df['StepTotal_scaled'] = scaler.fit_transform(df[['StepTotal']])

        logging.info(f'Transformación finalizada. Filas iniciales: {filas_iniciales}, finales: {len(df)}')
        return df
    except Exception as e:
        logging.error(f'Error en transformar_steps: {e}')
        raise

df_steps = transformar_steps(df_sucio)
print('Dimensiones después de limpieza:', df_steps.shape)
display(df_steps.head())
display(diagnostico_calidad(df_steps, 'df_steps_limpio'))

Dimensiones después de limpieza: (22179, 10)


,Id,ActivityHour,StepTotal,Hora,Fecha,Dia,Mes,FinDeSemana,PeriodoDia,StepTotal_scaled
0,1503960366,2016-03-12 00:00:00,0.0,0,2016-03-12,Saturday,3,1,madrugada,-0.419097
1,1503960366,2016-03-12 01:00:00,0.0,1,2016-03-12,Saturday,3,1,madrugada,-0.419097
2,1503960366,2016-03-12 02:00:00,0.0,2,2016-03-12,Saturday,3,1,madrugada,-0.419097
3,1503960366,2016-03-12 03:00:00,0.0,3,2016-03-12,Saturday,3,1,madrugada,-0.419097
4,1503960366,2016-03-12 04:00:00,0.0,4,2016-03-12,Saturday,3,1,madrugada,-0.419097


Dataset: df_steps_limpio
Filas duplicadas: 0


,columna,tipo_dato,nulos,porcentaje_nulos,unicos
0,Id,int64,0,0.0,34
1,ActivityHour,datetime64[ns],0,0.0,755
2,StepTotal,float64,0,0.0,2103
3,Hora,int32,0,0.0,24
4,Fecha,object,0,0.0,32
5,Dia,object,0,0.0,7
6,Mes,int32,0,0.0,2
7,FinDeSemana,int64,0,0.0,2
8,PeriodoDia,object,0,0.0,4
9,StepTotal_scaled,float64,0,0.0,2103


## 7. Optimización para grandes volúmenes
Se agrega procesamiento por chunks, útil cuando el archivo crece y no conviene cargarlo completo en memoria.

In [11]:
# 11. Procesamiento por chunks
def procesar_csv_por_chunks(ruta_csv, chunk_size=5000):
    chunks_transformados = []
    total_chunks = 0
    for chunk in pd.read_csv(ruta_csv, chunksize=chunk_size):
        total_chunks += 1
        chunk_limpio = transformar_steps(chunk)
        chunks_transformados.append(chunk_limpio)
    df_final = pd.concat(chunks_transformados, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=['Id', 'ActivityHour'], keep='first')
    logging.info(f'Procesamiento por chunks terminado. Chunks procesados: {total_chunks}')
    return df_final

df_steps_chunks = procesar_csv_por_chunks(ruta_csv_sucio, chunk_size=5000)
print('Filas procesadas por chunks:', len(df_steps_chunks))

Filas procesadas por chunks: 22179


## 8. Integración de las tres fuentes
Se integran los pasos limpios, el clima por hora y los datos de usuarios desde SQL.

In [12]:
# 12. Integración CSV + API + SQL
df_steps['ActivityHour'] = pd.to_datetime(df_steps['ActivityHour'], errors='coerce').dt.floor('H')
df_clima['ActivityHour'] = pd.to_datetime(df_clima['ActivityHour'], errors='coerce').dt.floor('H')

df_integrado = df_steps.merge(df_clima, on='ActivityHour', how='left')
df_integrado = df_integrado.merge(df_sql, on='Id', how='left')

df_integrado['segmento_usuario'] = df_integrado['segmento_usuario'].fillna('sin_segmento')
df_integrado['meta_pasos_hora'] = df_integrado['meta_pasos_hora'].fillna(500)
df_integrado['audiencia_operativa'] = df_integrado['audiencia_operativa'].fillna('seguimiento')

df_integrado['cumple_meta'] = (df_integrado['StepTotal'] >= df_integrado['meta_pasos_hora']).astype(int)
df_integrado['brecha_meta'] = df_integrado['StepTotal'] - df_integrado['meta_pasos_hora']
df_integrado['nivel_actividad'] = pd.cut(df_integrado['StepTotal'], bins=[-1, 0, 500, 1500, np.inf], labels=['sin_actividad', 'baja', 'media', 'alta']).astype(str)

print('Dataset integrado:', df_integrado.shape)
display(df_integrado.head())

Dataset integrado: (22179, 19)


,Id,ActivityHour,StepTotal,Hora,Fecha,Dia,Mes,FinDeSemana,PeriodoDia,StepTotal_scaled,temperature_2m,precipitation,fuente_clima,segmento_usuario,meta_pasos_hora,audiencia_operativa,cumple_meta,brecha_meta,nivel_actividad
0,1503960366,2016-03-12 00:00:00,0.0,0,2016-03-12,Saturday,3,1,madrugada,-0.419097,19.1,0.0,api_open_meteo,medio,1000,riesgo_inactividad,0,-1000.0,sin_actividad
1,1503960366,2016-03-12 01:00:00,0.0,1,2016-03-12,Saturday,3,1,madrugada,-0.419097,19.0,0.0,api_open_meteo,medio,1000,riesgo_inactividad,0,-1000.0,sin_actividad
2,1503960366,2016-03-12 02:00:00,0.0,2,2016-03-12,Saturday,3,1,madrugada,-0.419097,18.8,0.0,api_open_meteo,medio,1000,riesgo_inactividad,0,-1000.0,sin_actividad
3,1503960366,2016-03-12 03:00:00,0.0,3,2016-03-12,Saturday,3,1,madrugada,-0.419097,18.1,0.0,api_open_meteo,medio,1000,riesgo_inactividad,0,-1000.0,sin_actividad
4,1503960366,2016-03-12 04:00:00,0.0,4,2016-03-12,Saturday,3,1,madrugada,-0.419097,17.7,0.0,api_open_meteo,medio,1000,riesgo_inactividad,0,-1000.0,sin_actividad


In [13]:
# 13. Validación posterior a la integración
reglas_calidad = {
    'sin_nulos_claves': df_integrado[['Id', 'ActivityHour', 'StepTotal']].isna().sum().sum() == 0,
    'pasos_no_negativos': (df_integrado['StepTotal'] >= 0).all(),
    'sin_duplicados_clave': df_integrado.duplicated(subset=['Id', 'ActivityHour']).sum() == 0,
    'tiene_fuente_api': 'temperature_2m' in df_integrado.columns,
    'tiene_fuente_sql': 'segmento_usuario' in df_integrado.columns,
}

for regla, resultado in reglas_calidad.items():
    print(f"{regla}: {'OK' if resultado else 'REVISAR'}")

if not all(reglas_calidad.values()):
    raise ValueError('Existen reglas de calidad que no se cumplen.')

logging.info('Todas las reglas de calidad fueron aprobadas.')

sin_nulos_claves: OK
pasos_no_negativos: OK
sin_duplicados_clave: OK
tiene_fuente_api: OK
tiene_fuente_sql: OK


## 9. Carga del resultado final
Se guardan los resultados en CSV, SQLite y JSON de métricas.

In [14]:
# 14. Carga del dataset final
ruta_final_csv = DATA_PROCESSED / 'actividad_integrada_final.csv'
ruta_final_db = DATA_PROCESSED / 'actividad_analytics.db'
ruta_metricas = DATA_PROCESSED / 'metricas_generales.json'

df_integrado.to_csv(ruta_final_csv, index=False)

conn = sqlite3.connect(ruta_final_db)
df_integrado.to_sql('actividad_integrada', conn, if_exists='replace', index=False)
conn.close()

metricas_generales = {
    'fecha_ejecucion': datetime.now().isoformat(),
    'filas_finales': int(len(df_integrado)),
    'usuarios_unicos': int(df_integrado['Id'].nunique()),
    'pasos_totales': float(df_integrado['StepTotal'].sum()),
    'promedio_pasos_hora': float(df_integrado['StepTotal'].mean()),
    'porcentaje_cumplimiento_meta': float(df_integrado['cumple_meta'].mean() * 100),
    'fuentes_integradas': ['CSV', 'API REST', 'SQLite']
}

with open(ruta_metricas, 'w', encoding='utf-8') as f:
    json.dump(metricas_generales, f, ensure_ascii=False, indent=4)

print('Archivos generados:')
print(ruta_final_csv)
print(ruta_final_db)
print(ruta_metricas)
metricas_generales

Archivos generados:
/content/proyecto_etl_actividad/data/processed/actividad_integrada_final.csv
/content/proyecto_etl_actividad/data/processed/actividad_analytics.db
/content/proyecto_etl_actividad/data/processed/metricas_generales.json


{'fecha_ejecucion': '2026-06-19T21:21:25.006420',
 'filas_finales': 22179,
 'usuarios_unicos': 34,
 'pasos_totales': 6046832.0,
 'promedio_pasos_hora': 272.6377203661121,
 'porcentaje_cumplimiento_meta': 14.09441363451914,
 'fuentes_integradas': ['CSV', 'API REST', 'SQLite']}

## 10. Visualizaciones iniciales en el notebook
Estas gráficas validan los resultados antes de usar el dashboard completo.

In [15]:
# 15. Visualizaciones con Plotly
import plotly.express as px

actividad_hora = df_integrado.groupby('Hora', as_index=False)['StepTotal'].mean()
fig1 = px.line(actividad_hora, x='Hora', y='StepTotal', markers=True, title='Promedio de pasos por hora del día')
fig1.show()

actividad_periodo = df_integrado.groupby('PeriodoDia', as_index=False)['StepTotal'].mean()
fig2 = px.bar(actividad_periodo, x='PeriodoDia', y='StepTotal', title='Promedio de pasos por periodo del día')
fig2.show()

cumplimiento_segmento = df_integrado.groupby('segmento_usuario', as_index=False)['cumple_meta'].mean()
cumplimiento_segmento['cumple_meta'] = cumplimiento_segmento['cumple_meta'] * 100
fig3 = px.bar(cumplimiento_segmento, x='segmento_usuario', y='cumple_meta', title='Porcentaje de cumplimiento de meta por segmento')
fig3.show()

## 11. Generación de archivos profesionales del proyecto
Esta celda crea ETL modular, API, dashboard, tests, documentación, Docker y evidencia Git.

In [16]:
# 16. Crear archivos del proyecto

etl_code = r'''
import json, sqlite3, logging
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path(__file__).resolve().parents[1]
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
LOGS_DIR = BASE_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=LOGS_DIR / "etl_pipeline.log", level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

ESQUEMA_STEPS = {"Id": ["int64", "float64", "object"], "ActivityHour": ["object", "datetime64[ns]"], "StepTotal": ["int64", "float64", "object"]}

def validar_esquema(df, esquema, nombre_fuente="fuente"):
    errores = []
    for columna, tipos_permitidos in esquema.items():
        if columna not in df.columns:
            errores.append(f"Falta la columna obligatoria: {columna}")
        elif str(df[columna].dtype) not in tipos_permitidos:
            errores.append(f"Columna {columna} tiene tipo {df[columna].dtype}; esperado {tipos_permitidos}")
    if errores:
        logging.error(f"Errores de esquema en {nombre_fuente}: {errores}")
        raise ValueError(" | ".join(errores))
    return True

def extraer_csv(ruta):
    df = pd.read_csv(ruta)
    validar_esquema(df, ESQUEMA_STEPS, "csv_steps")
    return df

def extraer_clima_api(start_date="2016-03-12", end_date="2016-04-12", lat=-33.45, lon=-70.66):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": "temperature_2m,precipitation", "timezone": "America/Santiago"}
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        hourly = response.json().get("hourly", {})
        df = pd.DataFrame({"ActivityHour": hourly.get("time", []), "temperature_2m": hourly.get("temperature_2m", []), "precipitation": hourly.get("precipitation", [])})
        df["ActivityHour"] = pd.to_datetime(df["ActivityHour"], errors="coerce")
        df["fuente_clima"] = "api_open_meteo"
        return df
    except Exception as e:
        logging.warning(f"Fallo API REST, se usa fallback. Error: {e}")
        fechas = pd.date_range(start=start_date, end=end_date, freq="H")
        np.random.seed(42)
        return pd.DataFrame({"ActivityHour": fechas, "temperature_2m": np.random.normal(18, 5, len(fechas)).round(1), "precipitation": np.random.choice([0, 0, 0, 0.2, 1.5], len(fechas)), "fuente_clima": "fallback_generado"})

def crear_y_extraer_sql(df_base):
    ruta_db = DATA_RAW / "actividad_usuarios.db"
    conn = sqlite3.connect(ruta_db)
    ids = df_base["Id"].dropna().astype("int64").drop_duplicates().head(100)
    np.random.seed(42)
    df_usuarios = pd.DataFrame({"Id": ids, "segmento_usuario": np.random.choice(["bajo", "medio", "alto"], size=len(ids), p=[0.35, 0.45, 0.20]), "meta_pasos_hora": np.random.choice([250, 500, 750, 1000], size=len(ids)), "audiencia_operativa": np.random.choice(["seguimiento", "riesgo_inactividad", "usuario_activo"], size=len(ids))})
    df_usuarios.to_sql("usuarios", conn, if_exists="replace", index=False)
    df_sql = pd.read_sql_query("SELECT * FROM usuarios", conn)
    conn.close()
    return df_sql

def transformar_steps(df):
    df = df.copy()
    validar_esquema(df, ESQUEMA_STEPS, "steps_transformacion")
    df["Id"] = pd.to_numeric(df["Id"], errors="coerce")
    df["StepTotal"] = pd.to_numeric(df["StepTotal"], errors="coerce")
    df["ActivityHour"] = pd.to_datetime(df["ActivityHour"], errors="coerce")
    df = df.dropna(subset=["Id", "ActivityHour"])
    df["StepTotal"] = df["StepTotal"].fillna(0).clip(lower=0)
    df["Id"] = df["Id"].astype("int64")
    df = df.drop_duplicates(subset=["Id", "ActivityHour"], keep="first")
    df["Hora"] = df["ActivityHour"].dt.hour
    df["Fecha"] = df["ActivityHour"].dt.date.astype(str)
    df["Dia"] = df["ActivityHour"].dt.day_name()
    df["Mes"] = df["ActivityHour"].dt.month
    df["FinDeSemana"] = df["ActivityHour"].dt.dayofweek.isin([5, 6]).astype(int)
    df["PeriodoDia"] = pd.cut(df["Hora"], bins=[-1, 5, 11, 17, 23], labels=["madrugada", "mañana", "tarde", "noche"]).astype(str)
    df["StepTotal_scaled"] = StandardScaler().fit_transform(df[["StepTotal"]])
    return df

def integrar_fuentes(df_steps, df_clima, df_sql):
    df_steps["ActivityHour"] = pd.to_datetime(df_steps["ActivityHour"]).dt.floor("H")
    df_clima["ActivityHour"] = pd.to_datetime(df_clima["ActivityHour"]).dt.floor("H")
    df = df_steps.merge(df_clima, on="ActivityHour", how="left").merge(df_sql, on="Id", how="left")
    df["segmento_usuario"] = df["segmento_usuario"].fillna("sin_segmento")
    df["meta_pasos_hora"] = df["meta_pasos_hora"].fillna(500)
    df["audiencia_operativa"] = df["audiencia_operativa"].fillna("seguimiento")
    df["cumple_meta"] = (df["StepTotal"] >= df["meta_pasos_hora"]).astype(int)
    df["brecha_meta"] = df["StepTotal"] - df["meta_pasos_hora"]
    df["nivel_actividad"] = pd.cut(df["StepTotal"], bins=[-1, 0, 500, 1500, np.inf], labels=["sin_actividad", "baja", "media", "alta"]).astype(str)
    return df

def cargar_resultados(df):
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_PROCESSED / "actividad_integrada_final.csv", index=False)
    conn = sqlite3.connect(DATA_PROCESSED / "actividad_analytics.db")
    df.to_sql("actividad_integrada", conn, if_exists="replace", index=False)
    conn.close()
    metricas = {"fecha_ejecucion": datetime.now().isoformat(), "filas_finales": int(len(df)), "usuarios_unicos": int(df["Id"].nunique()), "pasos_totales": float(df["StepTotal"].sum()), "promedio_pasos_hora": float(df["StepTotal"].mean()), "porcentaje_cumplimiento_meta": float(df["cumple_meta"].mean() * 100), "fuentes_integradas": ["CSV", "API REST", "SQLite"]}
    with open(DATA_PROCESSED / "metricas_generales.json", "w", encoding="utf-8") as f:
        json.dump(metricas, f, ensure_ascii=False, indent=4)
    return metricas

def ejecutar_pipeline():
    logging.info("Inicio pipeline ETL")
    df_csv = extraer_csv(DATA_RAW / "hourlySteps_sucio.csv")
    df_steps = transformar_steps(df_csv)
    df_clima = extraer_clima_api()
    df_sql = crear_y_extraer_sql(df_csv)
    df_final = integrar_fuentes(df_steps, df_clima, df_sql)
    metricas = cargar_resultados(df_final)
    logging.info("Pipeline ETL finalizado correctamente")
    return metricas

if __name__ == "__main__":
    print(ejecutar_pipeline())
'''

api_code = r'''
from pathlib import Path
import json
import pandas as pd
from fastapi import FastAPI, HTTPException

BASE_DIR = Path(__file__).resolve().parents[1]
DATA_PROCESSED = BASE_DIR / "data" / "processed"
app = FastAPI(title="API Actividad Física", description="API REST para consultar métricas del pipeline ETL", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "ok", "service": "api-actividad"}

@app.get("/metrics")
def get_metrics():
    ruta = DATA_PROCESSED / "metricas_generales.json"
    if not ruta.exists():
        raise HTTPException(status_code=404, detail="Ejecuta el ETL primero")
    return json.load(open(ruta, "r", encoding="utf-8"))

@app.get("/summary/hour")
def summary_hour():
    ruta = DATA_PROCESSED / "actividad_integrada_final.csv"
    if not ruta.exists():
        raise HTTPException(status_code=404, detail="Ejecuta el ETL primero")
    df = pd.read_csv(ruta)
    return {"data": df.groupby("Hora", as_index=False)["StepTotal"].mean().to_dict(orient="records")}

@app.get("/user/{user_id}")
def user_detail(user_id: int):
    ruta = DATA_PROCESSED / "actividad_integrada_final.csv"
    if not ruta.exists():
        raise HTTPException(status_code=404, detail="Ejecuta el ETL primero")
    df = pd.read_csv(ruta)
    usuario = df[df["Id"] == user_id]
    if usuario.empty:
        raise HTTPException(status_code=404, detail="Usuario no encontrado")
    return {"Id": user_id, "registros": int(len(usuario)), "pasos_totales": float(usuario["StepTotal"].sum()), "promedio_pasos_hora": float(usuario["StepTotal"].mean()), "cumplimiento_meta_promedio": float(usuario["cumple_meta"].mean() * 100)}
'''

dash_code = r'''
from pathlib import Path
import pandas as pd
import plotly.express as px
import streamlit as st

BASE_DIR = Path(__file__).resolve().parents[1]
DATA_PROCESSED = BASE_DIR / "data" / "processed"
st.set_page_config(page_title="Dashboard Actividad Física", layout="wide")
st.title("Dashboard de Actividad Física por Hora")
st.caption("Proyecto end-to-end con ETL, API, SQL, dashboard, Git y Docker")
ruta = DATA_PROCESSED / "actividad_integrada_final.csv"
if not ruta.exists():
    st.error("No existe el dataset final. Ejecuta primero el pipeline ETL.")
    st.stop()
df = pd.read_csv(ruta)
audiencia = st.sidebar.selectbox("Selecciona audiencia", ["Ejecutiva", "Técnica", "Operativa"])
segmentos = sorted(df["segmento_usuario"].dropna().unique())
segmento = st.sidebar.multiselect("Segmento", segmentos, default=segmentos)
df_filtrado = df[df["segmento_usuario"].isin(segmento)]

if audiencia == "Ejecutiva":
    st.header("Vista ejecutiva")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Usuarios únicos", f"{df_filtrado['Id'].nunique():,}")
    c2.metric("Pasos totales", f"{df_filtrado['StepTotal'].sum():,.0f}")
    c3.metric("Promedio pasos/hora", f"{df_filtrado['StepTotal'].mean():,.1f}")
    c4.metric("Cumplimiento meta", f"{df_filtrado['cumple_meta'].mean()*100:.1f}%")
    st.plotly_chart(px.bar(df_filtrado.groupby("PeriodoDia", as_index=False)["StepTotal"].mean(), x="PeriodoDia", y="StepTotal", title="Actividad promedio por periodo del día"), use_container_width=True)
    cumplimiento = df_filtrado.groupby("segmento_usuario", as_index=False)["cumple_meta"].mean()
    cumplimiento["cumple_meta"] = cumplimiento["cumple_meta"] * 100
    st.plotly_chart(px.bar(cumplimiento, x="segmento_usuario", y="cumple_meta", title="Cumplimiento de meta por segmento (%)"), use_container_width=True)
elif audiencia == "Técnica":
    st.header("Vista técnica")
    st.write("Dimensiones del dataset:", df_filtrado.shape)
    st.dataframe(df_filtrado.isna().sum().reset_index().rename(columns={"index": "columna", 0: "nulos"}))
    st.plotly_chart(px.histogram(df_filtrado, x="StepTotal", nbins=50, title="Distribución de pasos por hora"), use_container_width=True)
    fuentes = df_filtrado["fuente_clima"].value_counts().reset_index()
    fuentes.columns = ["fuente_clima", "registros"]
    st.plotly_chart(px.pie(fuentes, names="fuente_clima", values="registros", title="Origen de datos de clima"), use_container_width=True)
else:
    st.header("Vista operativa")
    st.plotly_chart(px.line(df_filtrado.groupby("Hora", as_index=False)["StepTotal"].mean(), x="Hora", y="StepTotal", markers=True, title="Actividad promedio por hora"), use_container_width=True)
    ranking = df_filtrado.groupby("Id", as_index=False).agg(pasos_totales=("StepTotal", "sum"), cumplimiento=("cumple_meta", "mean"), registros=("StepTotal", "count")).sort_values("pasos_totales", ascending=False).head(20)
    ranking["cumplimiento"] = ranking["cumplimiento"] * 100
    st.dataframe(ranking)
'''

test_code = r'''
from pathlib import Path
import pandas as pd
BASE_DIR = Path(__file__).resolve().parents[1]
DATA_PROCESSED = BASE_DIR / "data" / "processed"

def test_dataset_final_existe():
    assert (DATA_PROCESSED / "actividad_integrada_final.csv").exists()

def test_columnas_obligatorias():
    df = pd.read_csv(DATA_PROCESSED / "actividad_integrada_final.csv")
    for columna in ["Id", "ActivityHour", "StepTotal", "Hora", "segmento_usuario", "cumple_meta"]:
        assert columna in df.columns

def test_pasos_no_negativos():
    df = pd.read_csv(DATA_PROCESSED / "actividad_integrada_final.csv")
    assert (df["StepTotal"] >= 0).all()

def test_sin_nulos_claves():
    df = pd.read_csv(DATA_PROCESSED / "actividad_integrada_final.csv")
    assert df[["Id", "ActivityHour", "StepTotal"]].isna().sum().sum() == 0
'''

readme = '''# Proyecto ETL Actividad Física

Solución end-to-end para analizar actividad física por hora. Integra CSV, API REST y SQLite, ejecuta un pipeline ETL robusto, expone resultados mediante FastAPI y visualiza indicadores en Streamlit.

## Fuentes
1. CSV: hourlySteps_sucio.csv.
2. API REST: Open-Meteo.
3. SQLite: actividad_usuarios.db.

## Ejecución
```bash
pip install -r requirements.txt
python etl/etl_pipeline.py
uvicorn api.main:app --host 0.0.0.0 --port 8000
streamlit run dashboards/app.py
```

## Docker
```bash
cd docker
docker compose up --build
```
'''

arquitectura = '''# Arquitectura técnica

```mermaid
flowchart LR
A[CSV] --> E[Pipeline ETL]
B[API REST Clima] --> E
C[SQLite Usuarios] --> E
E --> D[(Dataset Analítico)]
D --> F[API FastAPI]
D --> G[Dashboard Streamlit]
E --> H[Logs y métricas]
```

## Decisiones técnicas
- Pandas para transformación.
- SQLite para reproducibilidad.
- FastAPI para endpoints.
- Streamlit y Plotly para dashboard.
- Docker Compose para orquestación.
'''

api_doc = '''# Documentación de API

Base local: http://localhost:8000

Endpoints:
- GET /health
- GET /metrics
- GET /summary/hour
- GET /user/{user_id}

Swagger: http://localhost:8000/docs
'''

manual = '''# Manual de usuario

1. Ejecutar el pipeline ETL.
2. Abrir el dashboard.
3. Elegir audiencia: Ejecutiva, Técnica u Operativa.
4. Aplicar filtros por segmento.
5. Interpretar KPIs, gráficos y tablas.
'''

despliegue = '''# Guía de despliegue

Requisitos: Python 3.10, Docker y Docker Compose.

```bash
cp docker/.env.example docker/.env
cd docker
docker compose up --build
```

Servicios:
- API: http://localhost:8000
- Dashboard: http://localhost:8501
'''

guion = '''# Guion de presentación individual

1. Presentar objetivo del proyecto.
2. Explicar arquitectura: CSV, API, SQL, ETL, API y dashboard.
3. Mostrar validaciones, logging y manejo de errores.
4. Mostrar dashboard por audiencia y valor de negocio.
5. Explicar Git: ramas, commits, issues y pull requests.
6. Explicar Docker y despliegue.
7. Cerrar con lecciones aprendidas y mejoras futuras.
'''

git_workflow = '''# Flujo profesional de Git

Ramas sugeridas:
- main
- develop
- feature/etl-pipeline
- feature/api-rest
- feature/dashboard
- feature/docker-deploy
- docs/documentacion

Comandos:
```bash
git init
git checkout -b develop
git add .
git commit -m "chore: estructura inicial del proyecto"
git checkout -b feature/etl-pipeline
git commit -m "feat: agrega pipeline ETL con validación de esquemas"
```

Pull Request sugerido: feat: integra pipeline ETL con tres fuentes de datos.
Issues sugeridas: ETL, API, dashboard, Docker y documentación.
'''

requirements = '''pandas
numpy
requests
plotly
streamlit
fastapi
uvicorn
sqlalchemy
python-dotenv
scikit-learn
pytest
'''

dockerfile_api = '''FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

dockerfile_dashboard = '''FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "dashboards/app.py", "--server.address=0.0.0.0", "--server.port=8501"]
'''

docker_compose = '''services:
  api:
    build:
      context: ..
      dockerfile: docker/Dockerfile.api
    ports:
      - "8000:8000"
    env_file:
      - .env
    volumes:
      - ../data:/app/data
      - ../logs:/app/logs
  dashboard:
    build:
      context: ..
      dockerfile: docker/Dockerfile.dashboard
    ports:
      - "8501:8501"
    env_file:
      - .env
    volumes:
      - ../data:/app/data
      - ../logs:/app/logs
    depends_on:
      - api
'''

env_example = '''APP_ENV=development
API_HOST=0.0.0.0
API_PORT=8000
DASHBOARD_PORT=8501
DATA_PATH=/app/data
LOG_LEVEL=INFO
'''

# Escritura de archivos
(ETL_DIR / 'etl_pipeline.py').write_text(etl_code, encoding='utf-8')
(API_DIR / 'main.py').write_text(api_code, encoding='utf-8')
(DASH_DIR / 'app.py').write_text(dash_code, encoding='utf-8')
(TESTS_DIR / 'test_data_quality.py').write_text(test_code, encoding='utf-8')
(PROJECT_ROOT / 'requirements.txt').write_text(requirements, encoding='utf-8')
(PROJECT_ROOT / 'README.md').write_text(readme, encoding='utf-8')
(DOCS_DIR / 'README.md').write_text(readme, encoding='utf-8')
(DOCS_DIR / 'arquitectura.md').write_text(arquitectura, encoding='utf-8')
(DOCS_DIR / 'api_documentacion.md').write_text(api_doc, encoding='utf-8')
(DOCS_DIR / 'manual_usuario.md').write_text(manual, encoding='utf-8')
(DOCS_DIR / 'guia_despliegue.md').write_text(despliegue, encoding='utf-8')
(DOCS_DIR / 'guion_presentacion.md').write_text(guion, encoding='utf-8')
(DOCKER_DIR / 'Dockerfile.api').write_text(dockerfile_api, encoding='utf-8')
(DOCKER_DIR / 'Dockerfile.dashboard').write_text(dockerfile_dashboard, encoding='utf-8')
(DOCKER_DIR / 'docker-compose.yml').write_text(docker_compose, encoding='utf-8')
(DOCKER_DIR / '.env.example').write_text(env_example, encoding='utf-8')
(REPO_DIR / 'git_workflow.md').write_text(git_workflow, encoding='utf-8')

pd.DataFrame({
    'rama': ['feature/etl-pipeline', 'feature/api-rest', 'feature/dashboard', 'feature/docker-deploy', 'docs/documentacion'],
    'commit_sugerido': [
        'feat: agrega pipeline ETL con validación de esquemas',
        'feat: agrega API REST para métricas',
        'feat: agrega dashboard Streamlit por audiencia',
        'feat: agrega Docker y docker-compose',
        'docs: agrega documentación técnica y guía de despliegue'
    ],
    'responsable': ['Integrante 1', 'Integrante 2', 'Integrante 3', 'Integrante 4', 'Todo el equipo'],
    'evidencia': ['Pull request', 'Pull request', 'Pull request', 'Pull request', 'README y docs']
}).to_csv(REPO_DIR / 'evidencia_commits.csv', index=False)

print('Archivos profesionales creados correctamente')

Archivos profesionales creados correctamente


## 12. Testing automatizado
Ejecuta esta celda para validar que el dataset final cumple reglas mínimas de calidad.

In [17]:
# 17. Ejecutar pruebas
!cd /content/proyecto_etl_actividad && pytest -q tests

....                                                                     [100%]
4 passed in 0.76s


## 13. Comandos para la demo
Usa estos comandos en local o dentro del contenedor para demostrar el funcionamiento.

In [18]:
# 18. Comandos útiles
print('1) Ejecutar ETL:')
print('python etl/etl_pipeline.py')
print('\n2) Ejecutar API:')
print('uvicorn api.main:app --host 0.0.0.0 --port 8000')
print('\n3) Ejecutar dashboard:')
print('streamlit run dashboards/app.py')
print('\n4) Ejecutar Docker:')
print('cd docker && docker compose up --build')

1) Ejecutar ETL:
python etl/etl_pipeline.py

2) Ejecutar API:
uvicorn api.main:app --host 0.0.0.0 --port 8000

3) Ejecutar dashboard:
streamlit run dashboards/app.py

4) Ejecutar Docker:
cd docker && docker compose up --build


## 14. Exportar proyecto completo
Esta celda genera un ZIP con toda la estructura del proyecto para entregar o subir a GitHub.

In [19]:
# 19. Crear ZIP descargable del proyecto
zip_path = shutil.make_archive('/content/proyecto_etl_actividad', 'zip', PROJECT_ROOT)
print('ZIP creado:', zip_path)
print('Para descargarlo en Colab: panel izquierdo > Archivos > botón derecho sobre proyecto_etl_actividad.zip > Descargar')


ZIP creado: /content/proyecto_etl_actividad.zip
Para descargarlo en Colab: panel izquierdo > Archivos > botón derecho sobre proyecto_etl_actividad.zip > Descargar


## Checklist final antes de entregar

- El ETL corre sin errores.
- El dataset final existe en `data/processed/actividad_integrada_final.csv`.
- Los tests pasan correctamente.
- El dashboard abre y cambia entre audiencia ejecutiva, técnica y operativa.
- La API responde en `/health`, `/metrics`, `/summary/hour` y `/user/{id}`.
- La documentación explica arquitectura, API, instalación, despliegue y uso.
- Git tiene ramas, commits claros, issues, pull requests y evidencias.
- Docker tiene `Dockerfile.api`, `Dockerfile.dashboard`, `docker-compose.yml` y `.env.example`.